In [66]:
import pandas as pd
import numpy as np
import itertools

MutliFingerFlexion Adaptation:
Having participants produce finger extension or flexion of index, ring, or both. In each block, one condition is produced. A random cursor purturbation with zero mean and a certain variance will be added. Chords are chosen randomly. They should flex at -2 N or extend at +2N.

In [ ]:
fingers = [2, 4]

one_finger_chords = [[f] for f in fingers]
two_finger_chords = list(itertools.combinations(fingers, 2))

ext_flex_amp = [-2, 2]

# All 8 chord conditions: single-finger flex/extend + two-finger force combos
all_conditions = []
for chord in one_finger_chords:
    for f in ext_flex_amp:
        all_conditions.append((tuple[int, ...](chord), f, None))
for chord in two_finger_chords:
    for f1 in ext_flex_amp:
        for f2 in ext_flex_amp:
            all_conditions.append((chord, f1, f2))

iti = 1000 # msecs for inter-trial interval
planTime = 2000 # msecs for precue time
feedbackTime = 2000 # msecs for feedback time

purturbation_std = 0.5 # Newton

total_sub_num = 16
legacy_sub_nums = set(range(1, 11))  # already collected with old target files
skip_legacy_sub_nums = {7}  # sn7 file was patched; exclude from legacy reconstruction
num_trials_baseline = 20
num_trials_per_block = 120


In [68]:
all_conditions

[((2,), -2, None),
 ((2,), 2, None),
 ((4,), -2, None),
 ((4,), 2, None),
 ((2, 4), -2, -2),
 ((2, 4), -2, 2),
 ((2, 4), 2, -2),
 ((2, 4), 2, 2)]

In [ ]:
def legacy_condition_order(sub_num):
    """8-chord order from the original finger-group shuffle (participants 1-10)."""
    legacy_chords = one_finger_chords + two_finger_chords
    np.random.seed(sub_num - 1)
    chord_order = legacy_chords.copy()
    np.random.shuffle(chord_order)

    order = []
    for chord in chord_order:
        if len(chord) == 1:
            for f in ext_flex_amp:
                order.append((tuple(chord), f, None))
        else:
            for f1 in ext_flex_amp:
                for f2 in ext_flex_amp:
                    order.append((chord, f1, f2))
    return tuple(order)


def condition_order_from_seed(seed):
    np.random.seed(seed)
    order = all_conditions.copy()
    np.random.shuffle(order)
    return tuple(order)


def legacy_orders_by_sub(legacy_sub_nums, skip_sub_nums=None):
    """Map each legacy participant to their 8-chord order (dict preserves sub_num)."""
    skip_sub_nums = skip_sub_nums or set()
    return {
        sub_num: legacy_condition_order(sub_num)
        for sub_num in sorted(legacy_sub_nums)
        if sub_num not in skip_sub_nums
    }


def assign_shuffle_seed(sub_num, excluded_orders, start_seed=11):
    seed = start_seed
    while True:
        order = condition_order_from_seed(seed)
        if order not in excluded_orders:
            excluded_orders.add(order)
            return seed, list(order)
        seed += 1


def condition_label(chord, target_force1, target_force2):
    if target_force2 is None:
        finger = 'index' if chord[0] == 2 else 'ring'
        direction = 'flex' if target_force1 < 0 else 'extend'
        return f'{finger} {direction}'
    return f'both ({target_force1}, {target_force2})'

In [73]:
legacy_orders = legacy_orders_by_sub(legacy_sub_nums, skip_legacy_sub_nums)
excluded_orders = set(legacy_orders.values())
shuffle_seed_by_sub = {}

print(" ****************** new order ****************** \n")

for sub_num in range(1, total_sub_num + 1):
    if sub_num in legacy_sub_nums:
        continue
    seed, order = assign_shuffle_seed(sub_num, excluded_orders)
    shuffle_seed_by_sub[sub_num] = seed
    print(f'sn{sub_num} (shuffle seed {seed}):', ' -> '.join(condition_label(*c) for c in order))

 ****************** new order ****************** 

sn11 (shuffle seed 11): ring flex -> both (2, -2) -> both (-2, -2) -> both (-2, 2) -> both (2, 2) -> ring extend -> index flex -> index extend
sn12 (shuffle seed 12): both (-2, -2) -> both (2, -2) -> index flex -> ring flex -> index extend -> both (-2, 2) -> both (2, 2) -> ring extend
sn13 (shuffle seed 13): index extend -> both (-2, -2) -> ring extend -> both (-2, 2) -> both (2, -2) -> both (2, 2) -> index flex -> ring flex
sn14 (shuffle seed 14): both (-2, 2) -> both (2, -2) -> both (2, 2) -> ring flex -> index extend -> both (-2, -2) -> index flex -> ring extend
sn15 (shuffle seed 15): ring flex -> ring extend -> index extend -> both (2, -2) -> both (2, 2) -> both (-2, -2) -> both (-2, 2) -> index flex
sn16 (shuffle seed 16): ring flex -> both (-2, -2) -> index flex -> ring extend -> both (-2, 2) -> both (2, 2) -> both (2, -2) -> index extend


In [74]:
print(" ****************** old order ****************** ")
for sub_num, order in legacy_orders.items():
    print(f'sn{sub_num}:', ' -> '.join(condition_label(*c) for c in order))

 ****************** old order ****************** 
sn1: both (-2, -2) -> both (-2, 2) -> both (2, -2) -> both (2, 2) -> ring flex -> ring extend -> index flex -> index extend
sn2: index flex -> index extend -> both (-2, -2) -> both (-2, 2) -> both (2, -2) -> both (2, 2) -> ring flex -> ring extend
sn3: both (-2, -2) -> both (-2, 2) -> both (2, -2) -> both (2, 2) -> ring flex -> ring extend -> index flex -> index extend
sn4: ring flex -> ring extend -> index flex -> index extend -> both (-2, -2) -> both (-2, 2) -> both (2, -2) -> both (2, 2)
sn5: ring flex -> ring extend -> index flex -> index extend -> both (-2, -2) -> both (-2, 2) -> both (2, -2) -> both (2, 2)
sn6: index flex -> index extend -> ring flex -> ring extend -> both (-2, -2) -> both (-2, 2) -> both (2, -2) -> both (2, 2)
sn8: both (-2, -2) -> both (-2, 2) -> both (2, -2) -> both (2, 2) -> ring flex -> ring extend -> index flex -> index extend
sn9: both (-2, -2) -> both (-2, 2) -> both (2, -2) -> both (2, 2) -> ring flex -> 

In [ ]:
### target file headers: [subNum, planTime, feedbackTime, iti, session, 
# targetForce1, targetForce2, targetForce3, targetForce4, targetForce5]


legacy_orders = legacy_orders_by_sub(legacy_sub_nums, skip_legacy_sub_nums)
excluded_orders = set(legacy_orders.values())
shuffle_seed_by_sub = {}

for sub_num in range(1, total_sub_num + 1):
    if sub_num in legacy_sub_nums:
        continue
    seed, order = assign_shuffle_seed(sub_num, excluded_orders)
    shuffle_seed_by_sub[sub_num] = seed
    print(f'sn{sub_num} (shuffle seed {seed}):', ' -> '.join(condition_label(*c) for c in order))

for sub_num in range(1, total_sub_num + 1):
    if sub_num in legacy_sub_nums:
        continue

    bn = 1
    np.random.seed(shuffle_seed_by_sub[sub_num])
    condition_order = all_conditions.copy()
    np.random.shuffle(condition_order)

    for chord, target_force1, target_force2 in condition_order:
        target_forces = np.zeros(5)

        if target_force2 is None:
            seed = hash((chord, target_force1, 0)) % (2**32 - 1)
            np.random.seed(seed)
            target_forces[chord[0] - 1] = target_force1
            noise = np.round(np.random.normal(0, purturbation_std, num_trials_per_block), 2)
            if chord[0] == 2:
                purturbation1, purturbation2 = noise, 0
            else:
                purturbation1, purturbation2 = 0, noise
        else:
            seed = hash((chord, target_force1, target_force2)) % (2**32 - 1)
            np.random.seed(seed)
            target_forces[chord[0] - 1] = target_force1
            target_forces[chord[1] - 1] = target_force2
            purturbation1 = np.round(np.random.normal(0, purturbation_std, num_trials_per_block), 2)
            purturbation2 = np.round(np.random.normal(0, purturbation_std, num_trials_per_block), 2)

        for block_type in ['baseline', 'purturbed']:
            block_data = pd.DataFrame(
                [[sub_num, target_forces[0], target_forces[1], target_forces[2], target_forces[3], target_forces[4], 1,
                  0, 0, planTime, feedbackTime, iti]],
                columns=['subNum', 'targetForce1', 'targetForce2', 'targetForce3', 'targetForce4', 'targetForce5', 'isTargetVisible',
                         'purturbation1', 'purturbation2', 'planTime', 'feedbackTime', 'iti'])

            if block_type == 'baseline':
                block_data = pd.concat([block_data] * num_trials_baseline, ignore_index=True)
            else:
                block_data = pd.concat([block_data] * num_trials_per_block, ignore_index=True)
                block_data['purturbation1'] = purturbation1
                block_data['purturbation2'] = purturbation2

            block_data.to_csv(f'sn{sub_num}_{bn}.tgt', index=False, sep='\t')
            bn += 1

sn11 (shuffle seed 11): ring flex -> both (2, -2) -> both (-2, -2) -> both (-2, 2) -> both (2, 2) -> ring extend -> index flex -> index extend
sn12 (shuffle seed 12): both (-2, -2) -> both (2, -2) -> index flex -> ring flex -> index extend -> both (-2, 2) -> both (2, 2) -> ring extend
sn13 (shuffle seed 13): index extend -> both (-2, -2) -> ring extend -> both (-2, 2) -> both (2, -2) -> both (2, 2) -> index flex -> ring flex
sn14 (shuffle seed 14): both (-2, 2) -> both (2, -2) -> both (2, 2) -> ring flex -> index extend -> both (-2, -2) -> index flex -> ring extend
sn15 (shuffle seed 15): ring flex -> ring extend -> index extend -> both (2, -2) -> both (2, 2) -> both (-2, -2) -> both (-2, 2) -> index flex
sn16 (shuffle seed 16): ring flex -> both (-2, -2) -> index flex -> ring extend -> both (-2, 2) -> both (2, 2) -> both (2, -2) -> index extend
